# Learning From Random Projections (LFR) on the DAGHAR Dataset

In this notebook, we will pre-train a CNN_PF encoder using LFR on the DAGHAR KuHAR dataset, then fine-tune it on the same dataset. Finally, we will evaluate the model's performance using a test set. Note that while this example performs both pre-training and fine-tuning on the same dataset, in practical scenarios, pre-training typically involves a larger, more diverse dataset.

The following core libraries from `minerva` will be used:

1. **Pre-Training**:  
   - [`minerva.models.ssl.cpc.CPC`](): Implements CPC as a PyTorch Lightning module.
   - [`minerva.models.nets.tnc.TSEncoder`](): Implements the TS2Vec encoder (our "backbone"), which will be trained using CPC (`g_enc`).
   - [`minerva.models.nets.cpc_networks.HARCPCAutoregressive`](): Implements the CPC autoregressive network (`g_ar`), the default autoregressive network used in CPC for human activity recognition.
   - [`minerva.data.data_modules.har_rodrigues_24.HARDataModuleCPC`](): A Lightning DataModule that loads and organizes the DAGHAR KuHAR dataset for CPC training, providing training, validation, and test data loaders.
   - [`minerva.pipelines.lightning_pipeline.SimpleLightningPipeline`](): A wrapper for PyTorch Lightning’s `fit` method, enhancing reproducibility, logging, and customization for analytical purposes.
2. **Fine-Tuning**:
   - [`minerva.models.nets.base.SimpleSupervisedModel`](): Implements a supervised model with a customizable backbone (the pre-trained TS2Vec encoder) and head. This setup allows backbone freezing and supports a flexible head structure.
      - `FromPretrained`: Loads only the backbone (TS2Vec) from the checkpoint saved during pre-training.
      - [`minerva.models.nets.mlp.MLP`](): Defines a simple MLP classifier used as the head of the supervised model.
   - [`minerva.pipelines.lightning_pipeline.SimpleLightningPipeline`](): The same pipeline wrapper used in pre-training, with additional support for custom evaluation metrics and analysis.
   - [`minerva.data.data_modules.har.MultiModalHARSeriesDataModule`](): A Lightning DataModule for loading data in the format required for fine-tuning, organized similarly to the DAGHAR dataset. This module provides sliding window time series data suitable for fine-tuning.

**Note**: 
1. Although we use the DAGHAR dataset here, this pipeline is flexible and can be adapted for other datasets. 
2. The data module configuration differs between pre-training (using full time series for each sample) and fine-tuning (using sliding windows) to suit model requirements at each stage.

### Useful Links:
- [DAGHAR Dataset on Zenodo](https://zenodo.org/records/13987073)
- [Contrastive Predictive Coding for Human Activity Recognition]()
- [TS2Vec: Towards Universal Representation of Time Series](https://cdn.aaai.org/ojs/20881/20881-13-24894-1-2-20220628.pdf)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [2]:
from datetime import datetime

import lightning as L
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from minerva.data.data_modules.har_rodrigues_24 import HARDataModuleCPC

from minerva.pipelines.lightning_pipeline import SimpleLightningPipeline
from minerva.models.nets.base import SimpleSupervisedModel

from minerva.models.nets.tnc import TSEncoder
import torchmetrics

from minerva.data.data_modules.har import MultiModalHARSeriesDataModule
from minerva.models.loaders import FromPretrained
from minerva.models.nets.base import SimpleSupervisedModel
from minerva.models.nets.mlp import MLP
from minerva.analysis.metrics.balanced_accuracy import BalancedAccuracy
from minerva.analysis.model_analysis import TSNEAnalysis
from minerva.models.nets.tnc import RnnEncoder

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
execution_id = f'run_{datetime.now().strftime("%Y%m%d-%H%M%S")}'
log_dir = f"./logs/{execution_id}" 

print(f"Execution ID: {execution_id}")
print(f"Log dir: {log_dir}")

Execution ID: run_20250516-172317
Log dir: ./logs/run_20250516-172317


## Pre-training with LFR

We will train a TS2Vec Encoder using [DAGHAR's KuHAr dataset](https://zenodo.org/records/13987073) and Contrastive Predictive Coding (CPC) as the pretext task. 

### 1. Defining the Data Module

We will use the `HARDataModuleCPC` data module to load the DAGHAR dataset for CPC training. This data module loads the data in the format required for CPC training, which includes full time series data for each sample.

Required arguments:
- `data_path`: Path to the directory containing the dataset.
- `input_size`: The number of features in the input time series.
- `window_size`: The size of the sliding window used to create the positive and negative samples for CPC.
- `overlap`: The overlap between consecutive windows.
- `batch_size`: The batch size for training.

In [4]:
# data_module = HARDataModuleCPC(
#     data_path="/workspaces/HIAAC-KR-Dev-Container/shared_data/rodrigues_2024_datasets/1-1/kuhar/",
#     input_size=6,
#     window=60,
#     overlap=60,
#     batch_size=64,
#     use_index_as_label=False,
#     use_val_with_train = True
# )

# data_module

In [5]:
# # Pega os dataloader de treino
# data_module.setup("fit")
# train_data_loader = data_module.train_dataloader()

# # Obtem o primeiro batch de treino (64 amostras de 6x60)
# first_batch = next(iter(train_data_loader))

# X, y = first_batch
# print(f"O primeiro batch de treino tem shape X={tuple(X.shape)} e y={tuple(y.shape)}")

In [6]:
# train_size = len(data_module.train_dataloader().dataset)
# print(f"Train size: {train_size}")

### 2. Defining the DIET Model

We will create the CPC model using the TS2Vec encoder as the backbone and the HARCPCAutoregressive network as the autoregressive network.

The TS2Vec encoder (`g_enc`) requires the following arguments:
- `input_dims`: The number of features in the input time series.
- `output_dims`: The dimensionality of the output embeddings.
- `depth`: Number of convolutional layers.
- `permute`: Whether to permute the input time series before passing it through the encoder.

The HARCPCAutoregressive network (`g_ar`) requires the following arguments:
- `input_size`: The dimensionality of the input embeddings.
- `hidden_size`: The dimensionality of the hidden state in the autoregressive network.
- `batch_first`: Whether the input is batch-first.
- `bidirectional`: Whether the autoregressive network is bidirectional.

The `CPC` model requires the following arguments:
- `g_enc`: The TS2Vec encoder.
- `g_ar`: The CPC autoregressive network.
- `prediction_head_in_channels`: The dimensionality of the input to the prediction head.
- `prediction_head_out_channels`: The dimensionality of the output of the prediction head.
- `num_steps_prediction`: The number of steps to predict in the future.
- `batch_size`: The batch size for training.
- `minimum_steps`: The minimum number of steps in the future to predict.

In [7]:
from minerva.models.nets.time_series.cnns import CNN_PF_Backbone
from minerva.models.ssl.lfr import LearnFromRandomnessModel
from minerva.models.nets.lfr_har_architectures import LFR_HAR_Predictor, LFR_HAR_Projector
from torch.nn import ModuleList

# Backbone
backbone = CNN_PF_Backbone(include_middle=True,flatten=True)
backbone_encoding_size = 768 # Output from the backbone

# LFR - from paper
lfr_learning_rate = 3e-4
lfr_weight_Decay = 3e-4
lfr_predictor_training_epochs = 5
lfr_total_projections = 6

lfr_projectors = ModuleList([LFR_HAR_Projector(encoding_size=backbone_encoding_size, input_channel=6) for _ in range(lfr_total_projections)])
lfr_predictors = ModuleList([LFR_HAR_Predictor(encoding_size=backbone_encoding_size, middle_dim=128, num_layers=1) for _ in range(lfr_total_projections)])

model = LearnFromRandomnessModel(
    backbone=backbone,
    projectors=lfr_projectors,
    predictors=lfr_predictors,
    loss_fn=None,
    flatten=False,
    predictor_training_epochs=lfr_predictor_training_epochs,
    learning_rate=lfr_learning_rate,
    weight_decay=lfr_weight_Decay
)
model

LearnFromRandomnessModel(
  (backbone): CNN_PF_Backbone(
    (first_padder): ZeroPadder2D(pad_at=(3,), padding_size=2)
    (upper_part): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=1, dilation=1, ceil_mode=False)
    )
    (lower_part): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=1, dilation=1, ceil_mode=False)
    )
    (middle_part): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=1, dilation=1, ceil_mode=False)
    )
    (shared_part): Sequential(
      (0): Conv2d(48, 64, kernel_size=(3, 5), stride=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=1, dilation=1, ceil_mode=False)
    )
  )
  (projectors): ModuleList(
    (0

### 3. Defining the Pytorch Lightning Trainer Configuration

We will define the PyTorch Lightning Trainer configuration for training the CPC model. This configuration includes the following parameters:
- `max_epochs`: The maximum number of epochs for training.
- `accelartor`: The device to use for training (e.g., 'cuda' or 'cpu').
- `devices`: The number (or a list) of accelerator devices to use.
- `logger`: The logger to use for logging training metrics.
- `callbacks`: The callbacks to use during training. Callbacks allows customizing the training loop and adding additional functionality, such as early stopping, model checkpointing, and learning rate scheduling.
- `limit_*_batches`: The number of batches to limit the training, validation, and testing datasets. This is useful for debugging and testing the pipeline.

The logger and callbacks can be customized based on the requirements of the training pipeline.
We will use the `CSVLogger` callback to log the training metrics to a CSV file and the `ModelCheckpoint` callback to save the best model based on the validation loss.

In [8]:
## Callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints/',
    monitor='train_loss',
    mode='min',
    save_last=True
)

## Logger
logger = CSVLogger(save_dir=log_dir, name='backbone-pretraining', version=execution_id)

## Trainer
trainer = L.Trainer(
    max_epochs=3,
    accelerator="gpu",
    devices=1,
    logger=logger,
    callbacks=[checkpoint_callback],
    # Only for testing. Remove for production. We will only train using 1 batch
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


### 4. Creating the training pipeline (and running the training

We will create a `SimpleLightningPipeline` to train the CPC model. This pipeline provides a high-level interface for training the model, logging the training metrics, and saving the best model based on the validation loss. Also, it allows customizing the evaluation metrics and analysis based on the requirements.

The pipeline requires the following arguments:
- `model`: The CPC model to train.
- `trainer`: The PyTorch Lightning Trainer configuration.
- `log_dir`: The directory to save the training logs and model checkpoints.
- `seed`: The random seed for reproducibility.
- `save_run_stats`: Whether to save the training statistics for reproducibility and analysis.

Once the pipeline is created, we can call the `run` method with the data module to start the training process. The parameter `task` of `run` method should be set to:
- `fit`: To train the model.
- `evaluate`: To evaluate the model on the test set.

In [9]:
train_pipeline = SimpleLightningPipeline(
    model=model,
    trainer=trainer,
    log_dir=log_dir,
    save_run_status=True,
    seed=42
)
# train_pipeline.run(data_module, task="fit")

### 5. Inspecting Checkpoints

Once the training is complete, we can inspect the saved checkpoints to load the pre-trained model for fine-tuning or further analysis.

Let's inspect the checkpoint file from the trained CPC model and check the weights that correspond to the TS2Vec encoder (`g_enc`).

If you check the output, the checkpoint keys that starts with `g_enc` are the weights of the TS2Vec encoder. The `g_ar` weights are also saved in the checkpoint, and also the `prediction_head` weights. It is important to note that the `g_ar` and `prediction_head` weights are not used in the fine-tuning process, as we will only load the TS2Vec encoder for fine-tuning. Thus, we will only load the weights corresponding to the `g_enc` keys.

In [10]:
# ckpt_path = checkpoint_callback.last_model_path
# ckpt = torch.load(ckpt_path, map_location="cpu")
# ckpt = ckpt.get("state_dict", ckpt)
# list(ckpt.keys())

## Fine-tuning the Pre-trained TS2Vec Encoder

After pre-training the TS2Vec encoder using CPC, we will fine-tune the encoder on the DAGHAR dataset for human activity recognition. We will use the same DAGHAR dataset but with a different data module configuration that provides sliding window time series data suitable for fine-tuning.

### 1. Defining the Data Module

We will use the `MultiModalHARSeriesDataModule` data module to load the DAGHAR dataset for fine-tuning. This data module loads the data in the format required for fine-tuning, which includes sliding window time series data for each sample. Thus, each sample of the dataset will be a 2-element tuple containing the time series (6x60, where 6 is the number of features and 60 is the window size) and the corresponding label.

The data module requires the following arguments:
- `data_path`: Path to the directory containing the dataset.
- `feature_prefix`: The prefix of the columns containing the features. For each prefix, we will create a different channel with all columns that start with the prefix.
- `label`: The name of the column containing the labels.
- `features_as_channels`: If True, for each prefix, we will create a different channel with all columns that start with the prefix. If False, we will concatenate all columns with the same prefix into a single channel (the sample will be a tensor of 1x360 instead of 6x60).
- `cast_to`: The data type to cast the features (float32).
- `batch_size`: The batch size for training.

In [11]:
data_module = MultiModalHARSeriesDataModule(
    data_path="/workspaces/HIAAC-KR-Dev-Container/shared_data/daghar/standardized_view/UCI/",
    feature_prefixes=["accel-x", "accel-y", "accel-z", "gyro-x", "gyro-y", "gyro-z"],
    label="standard activity code",
    features_as_channels=True,
    cast_to="float32",
    batch_size=64,
    channels_before_timestamps=False
)

data_module

MultiModalHARSeriesDataModule(data_path=/workspaces/HIAAC-KR-Dev-Container/shared_data/daghar/standardized_view/UCI, batch_size=64)

In [12]:
# Pega os dataloaders de treino e validação
data_module.setup("fit")
train_data_loader = data_module.train_dataloader()
validation_data_loader = data_module.val_dataloader()
first_batch = next(iter(train_data_loader))

X, y = first_batch
print(X.shape, y.shape)

Using DataLoader with shuffle=True
Using DataLoader with shuffle=False
torch.Size([64, 60, 6]) torch.Size([64])


In [13]:
# model_ = CNN_PF_Backbone(include_middle=True,flatten=True)

from DiffusionTS_utils import create_trainer
from torch import nn
import torch

class DiffusionTSWrapper(nn.Module):
    def __init__(self, model):
        super(DiffusionTSWrapper, self).__init__()
        self.model = model
        self.model.eval()  # Set the model to evaluation mode
        # self.model.freeze()
        self.model.requires_grad_(False)

    def forward(self, x):
        self.model.eval()
        timestep = 0
        timestep_tensor = torch.ones(len(x), dtype=torch.long)
        timestep_tensor = timestep_tensor.to("cuda") * timestep
        with torch.no_grad():
            embedding = self.model.model.emb(x)
            embedding = self.model.model.pos_enc(embedding)
            embedding = self.model.model.encoder(embedding, timestep_tensor)
        return embedding

folder = "D_UCI-trained_on-class_1-minmax_scaled-480000_epochs"
fake_dl_info = {
    "dataset": None,
    "test_dataloader": None,
    "dataloader": None,
}

trainer = create_trainer(folder=folder, dl_info=fake_dl_info, max_epochs=480000, num_checkpoints=10)
try:
    trainer.load(milestone=10, verbose=True)
    print("Checkpoint loaded successfully.")
except Exception as e:
    print(f"Error loading checkpoint: {e}")
    raise e

backbone = DiffusionTSWrapper(trainer.model)

Checkpoint loaded successfully.


### 2. Defining the Fine-tuning Model

We first load the pre-trained TS2Vec encoder from the checkpoint saved during pre-training. We then create the supervised model with the TS2Vec encoder as the backbone and an MLP classifier as the head.

To load the TS2Vec encoder from the checkpoint, we use the `FromPretrained` class, which loads only the backbone from the checkpoint. The `FromPretrained` class requires the following arguments:
- `model`: The encoder model, with randomly initialized weights.
- `ckpt_path`: The path to the checkpoint file.
- `filter_keys`: A list of keys to filter from checkpoint keys. We will use this argument to load only the weights corresponding to the TS2Vec encoder (The keys that start with `g_enc`).

In [14]:
# # g_enc = TSEncoder(input_dims=6, output_dims=64, hidden_dims=64, depth=10, permute=True)
# backbone = FromPretrained(
#     model=model_,#g_enc,
#     ckpt_path=checkpoint_callback.best_model_path,
#     filter_keys=["backbone"],
#     keys_to_rename={"backbone.": ""}, 
#     # keys_to_rename= { "g_enc.":"","g_ar.":""},
#     strict=True,
#     error_on_missing_keys=True
# )

Once backbone is loaded, we create the supervised model with the TS2Vec encoder as the backbone and an MLP classifier as the head. The MLP classifier requires the following arguments:
- A list with layer sizes for the MLP. The first element should be the size of the input layer (the output of the TS2Vec encoder), and the last element should be the size of the output layer (the number of classes). We use a MLP with 3840 input units (the output of the TS2Vec encoder), a hidden layer with 128 units, and an output layer with 6 units (the number of classes in the DAGHAR dataset).

In [15]:
head = MLP([64*60, 128, 6])

Finally, we use the `SimpleSupervisedModel` class to create the supervised model with the TS2Vec encoder as the backbone and the MLP classifier as the head. The `SimpleSupervisedModel` class requires the following arguments:
- `backbone`: The backbone model (the pre-trained TS2Vec encoder).
- `fc`: The head model (the MLP classifier).
- `loss_fn`: The loss function to use for training (CrossEntropyLoss).
- `flatten`: Whether to flatten the input before passing it through the head. Usually, the input is flattened if the backbone outputs a tensor with more than two dimensions.
- `train_metrics`: A dictionary where the keys are the names of the metrics and the values are the functions that calculate the metrics. The metrics function should use `torchmetrics` API to calculate the metrics.
- `val_metrics`: A dictionary where the keys are the names of the metrics and the values are the functions that calculate the metrics. The metrics function should use `torchmetrics` API to calculate the metrics.

In [16]:
# model = SimpleSupervisedModel(
#     backbone=backbone,
#     fc=head,
#     loss_fn=torch.nn.CrossEntropyLoss(),
#     flatten=False,
#     train_metrics={
#         "acc": torchmetrics.Accuracy(task="multiclass", num_classes=6),
#     },
#     val_metrics={
#         "acc": torchmetrics.Accuracy(task="multiclass", num_classes=6),
#     },
# )

# model


# from minerva.models.adapters import MaxPoolingTransposingSqueezingAdapter

# adapter = MaxPoolingTransposingSqueezingAdapter(kernel_size=64)
model = SimpleSupervisedModel(
    backbone=backbone,
    fc=head,
    loss_fn=torch.nn.CrossEntropyLoss(),
    flatten=True,
    freeze_backbone=True,
    # adapter = adapter,
    train_metrics={
        "acc": torchmetrics.Accuracy(task="multiclass", num_classes=6),
    },
    val_metrics={
        "acc": torchmetrics.Accuracy(task="multiclass", num_classes=6),
    },
)

### 3. Defining the Pytorch Lightning Trainer Configuration

We will define the PyTorch Lightning Trainer configuration for fine-tuning the model. Save as for the pre-training.

In [17]:
## Callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints/',
    monitor='val_loss',
    mode='min',
    save_last=True
)

## Logger
logger = CSVLogger(save_dir=log_dir, name='model-finetuning', version=execution_id)

## Trainer
trainer = L.Trainer(
    max_epochs=100,
    accelerator="gpu",
    devices=1,
    logger=logger,
    callbacks=[checkpoint_callback],
    # Only for testing. Remove for production. We will only train using 1 batch
    # limit_train_batches=1,
    # limit_val_batches=1,
)

trainer

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### 4. Creating the fine-tuning pipeline (and running the training)

We will create a `SimpleLightningPipeline` to fine-tune the model. Save as for the pre-training.

In [18]:
train_pipeline = SimpleLightningPipeline(
    model=model,
    trainer=trainer,
    log_dir=log_dir,
    save_run_status=True,
    seed=42
)
train_pipeline.run(data_module, task="fit")

** Seed set to: 42 **
Pipeline info saved at: /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/logs/run_20250516-172317/run_2025-05-16-17-23-23314a5b2f.yaml


/usr/local/lib/python3.10/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [7]

  | Name     | Type               | Params | Mode 
--------------------------------------------------------
0 | backbone | DiffusionTSWrapper | 253 K  | train
1 | fc       | MLP                | 492 K  | train
2 | loss_fn  | CrossEntropyLoss   | 0      | train
--------------------------------------------------------
492 K     Trainable params
253 K     Non-trainable params
746 K     Total params
2.985     Total estimated model params size (MB)
7         Modules in train mode
119       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]Using DataLoader with shuffle=False
Using DataLoader with shuffle=True                                         


/usr/local/lib/python3.10/dist-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (37) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 99: 100%|██████████| 37/37 [00:09<00:00,  4.11it/s, v_num=2317, val_loss=0.359, val_acc=0.875, train_loss=0.137, train_acc=0.931]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 37/37 [00:09<00:00,  4.06it/s, v_num=2317, val_loss=0.359, val_acc=0.875, train_loss=0.137, train_acc=0.931]
Pipeline info saved at: /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/logs/run_20250516-172317/run_2025-05-16-17-23-23314a5b2f.yaml


## Evaluating the Fine-tuned Model

After fine-tuning the TS2Vec encoder on the DAGHAR dataset, we will evaluate the model's performance on the test set. We create a simple evaluation pipeline, that will:
1. Run forward on test set.
2. Calculcate the metrics. This is specified in the `classification_metrics` dictionary, where the keys are the names of the metrics and the values are the functions that calculate the metrics. The metrics function should use `torchmetrics` API to calculate the metrics.
3. Perform model analysis, such as plot t-sne embeddings.

The test pipeline requires the following arguments:
- `model`: The fine-tuned model to evaluate.
- `trainer`: The PyTorch Lightning Trainer configuration.
- `log_dir`: The directory to save the evaluation logs.
- `seed`: The random seed for reproducibility.
- `classification_metrics`: A dictionary where the keys are the names of the metrics and the values are the functions that calculate the metrics. The metrics function should use `torchmetrics` API to calculate the metrics.
- `model_analysis`: A function that performs model analysis, such as plotting t-sne embeddings.

Finally, we run the evaluation pipeline with the test data module to evaluate the model on the test set. We set the `task` parameter of the `run` method to `evaluate` to evaluate the model on the test set.

In [19]:
test_pipeline = SimpleLightningPipeline(
    model=model,
    trainer=trainer,
    log_dir=log_dir,
    save_run_status=True,
    # seed=42,
    classification_metrics={
        "accuracy": torchmetrics.Accuracy(num_classes=6, task="multiclass"),
        "f1": torchmetrics.F1Score(num_classes=6, task="multiclass"),
        "precision": torchmetrics.Precision(num_classes=6, task="multiclass"),
        "recall": torchmetrics.Recall(num_classes=6, task="multiclass"),
        "balanced_accuracy": BalancedAccuracy(num_classes=6, task="multiclass"),
    },
    apply_metrics_per_sample=False,
    # model_analysis={
    #     "tsne": TSNEAnalysis(
    #         height=800,
    #         width=800,
    #         legend_title="Activity",
    #         title="t-SNE of CPC Finetuned on MotionSense",
    #         output_filename="tsne_cpc_finetuned_motionsense.pdf",
    #         label_names={
    #             0: "sit",
    #             1: "stand",
    #             2: "walk",
    #             3: "stair up",
    #             4: "stair down",
    #             5: "run",
    #             6: "stair up and down",
    #         },
    #     )
    # },
)

test_pipeline.run(
    data_module, task="evaluate", ckpt_path=checkpoint_callback.best_model_path
)

** Seed set to: 42 **
Pipeline info saved at: /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/logs/run_20250516-172317/run_2025-05-16-17-39-446087efd5.yaml
Using DataLoader with shuffle=False


Restoring states from the checkpoint path at /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/checkpoints/epoch=19-step=740.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [7]
Loaded model weights from the checkpoint at /workspaces/HIAAC-KR-Dev-Container/Minerva-Dev/experiments/repos/Diffusion-TS/checkpoints/epoch=19-step=740.ckpt


Using DataLoader with shuffle=False
Predicting DataLoader 0: 100%|██████████| 11/11 [00:00<00:00, 101.32it/s]


ValueError: Unexpected keyword arguments: `compute_on_step`